# Tongsan Articles Cleaner — Kaggle Notebook

This notebook cleans Tongsan articles by:
1. **Removing all non-Zolai content** (Burmese-only articles, English-only articles)
2. **Stripping all HTML tags** (`<p>`, `<figure>`, `<img>`, `<em>`, `<strong>`, etc.)
3. **Removing WordPress noise** (`wp-block-image`, `size-large`, `is-resized`, etc.)
4. **Decoding HTML entities** (`&#8230;`, `&#8217;`, `&#8216;`, `&nbsp;`, etc.)
5. **Keeping only Zolai-language articles**

**Output:** Clean Zolai-only Tongsan articles as JSONL.

In [ ]:
# First cell: check internet and install deps
import os
import socket

try:
    socket.create_connection(("8.8.8.8", 53), timeout=3)
    HAS_INTERNET = True
except OSError:
    HAS_INTERNET = False

print(f"Internet: {'Connected' if HAS_INTERNET else 'Offline'}")

if HAS_INTERNET:
    !pip install html5lib pandas beautifulsoup4

In [ ]:
import json
import re
import html
import hashlib
from pathlib import Path
from collections import Counter

import pandas as pd
from bs4 import BeautifulSoup

# Paths — adjust for Kaggle dataset mount
INPUT_DIR = Path("/kaggle/input/zolai-dataset")
OUTPUT_DIR = Path("/kaggle/working")

# Fallback: if running locally
# INPUT_DIR = Path(r"data/zolai-focused")
# OUTPUT_DIR = Path(r"data/cleaned-tongsan")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Noise Removal Functions

In [ ]:
def decode_html_entities(text: str) -> str:
    """Decode all HTML entities (&#8230; &#8217; &nbsp; etc.)."""
    return html.unescape(text)


def strip_all_html(text: str) -> str:
    """Remove all HTML tags using BeautifulSoup for proper parsing."""
    try:
        soup = BeautifulSoup(text, "html.parser")
        return soup.get_text(separator=" ")
    except Exception:
        # Fallback to regex
        return re.sub(r'<[^>]+>', ' ', text)


def remove_wordpress_noise(text: str) -> str:
    """Remove WordPress-specific class names and block artifacts."""
    # Remove WordPress class names that leak into text
    wp_patterns = [
        r'wp-block-\S+',
        r'size-large',
        r'size-medium',
        r'size-thumbnail',
        r'is-resized',
        r'aligncenter',
        r'alignleft',
        r'alignright',
        r'alignnone',
        r'wp-image-\d+',
        r'attachment-\d+',
        r'srcset="[^"]*"',
        r'sizes="[^"]*"',
    ]
    for pattern in wp_patterns:
        text = re.sub(pattern, '', text)
    return text


def remove_image_urls(text: str) -> str:
    """Remove image URLs and media references."""
    text = re.sub(r'https?://\S+\.(jpg|jpeg|png|gif|webp|svg)\S*', '', text)
    text = re.sub(r'https?://\S+wp-content/uploads/\S*', '', text)
    return text


def normalize_whitespace(text: str) -> str:
    """Clean up whitespace: collapse multiple spaces, strip."""
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    # Fix spacing around punctuation
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    text = re.sub(r'([,.;:!?])\s*', r'\1 ', text).strip()
    return text


def clean_tongsan_text(text: str) -> str:
    """Full cleaning pipeline for Tongsan article text."""
    if not text or not isinstance(text, str):
        return ""
    
    # Step 1: Decode HTML entities
    text = decode_html_entities(text)
    
    # Step 2: Remove WordPress noise before HTML stripping
    text = remove_wordpress_noise(text)
    
    # Step 3: Remove image URLs
    text = remove_image_urls(text)
    
    # Step 4: Strip all HTML tags
    text = strip_all_html(text)
    
    # Step 5: Normalize whitespace
    text = normalize_whitespace(text)
    
    return text

## 2. Language Detection

In [ ]:
def has_burmese(text: str) -> bool:
    """Check if text contains Burmese script (Unicode U+1000 to U+109F)."""
    return any('\u1000' <= c <= '\u109F' for c in text)


def has_zolai_markers(text: str) -> bool:
    """Detect Zolai language using common Zolai words/particles.
    
    Zolai (Tedim/Chin dialect) has distinctive markers:
    - Sentence-final particles: hi, ci, ta, hiam, zong
    - Common words: Pasian, leh, mah, in, a, pen, om, bawl, khat
    - Grammar patterns: ci-in, hong, tung, kah
    """
    if not text:
        return False
    
    text_lower = text.lower()
    
    # Strong Zolai markers (sentence-final particles, unique words)
    strong_markers = [
        r'\shi\b', r'\sci\b', r'\sta\b', r'\shiam\b', r'\szong\b',
        r'\shi,', r'\sci,', r'\sta,',
        r'\shi\.', r'\sci\.', r'\sta\.',
        r'\spa\s', r'\spa\.', r'\spa,',
        r'ci\sin', r'ci\shi', r'ci\sta',
        r'hong\s', r'hong\.',
        r'\stkhuapa', r'\skhuapi', r'\skhua',
        r'\sngai', r'\sngei',
    ]
    
    # Common Zolai words
    zolai_words = [
        "pasian", "leh", "mah", "khat", "tung", "kah",
        "kei", "amah", "nite", "sun", "zan", "nitak", "zingsang",
        "vantung", "leitung", "tuipi", "khuavak", "khuamial",
        "bawl", "piang", "kum", "lai", "mun", "gam",
        "mihing", "galkap", "khuam", "thup", "thu",
        "ciangin", "cih", "cihna", "ahimanin", "tuate",
        "hanciam", "deihna", "lungdam", "minam", "ngeina",
        "tangthu", "kampau", "laih", "gelh", "sim",
    ]
    
    # Check strong markers first
    for pattern in strong_markers:
        if re.search(pattern, text_lower):
            return True
    
    # Count Zolai word occurrences
    zolai_count = sum(1 for word in zolai_words if word in text_lower)
    
    # Need at least 2 Zolai words to consider it Zolai
    return zolai_count >= 2


def classify_article(record: dict) -> str:
    """Classify an article as 'zolai', 'burmese', 'english', or 'mixed'."""
    content = record.get("content", "")
    title = record.get("title", "")
    full_text = title + " " + content
    
    is_zolai = has_zolai_markers(full_text)
    is_burmese = has_burmese(full_text)
    
    if is_zolai and is_burmese:
        return "mixed"
    elif is_zolai:
        return "zolai"
    elif is_burmese:
        return "burmese"
    else:
        return "english"

## 3. Process Tongsan Articles

In [ ]:
def process_tongsan_articles(input_path: Path, output_zolai: Path, output_stats: Path):
    """Process Tongsan articles: filter non-Zolai, clean noise, deduplicate."""
    if not input_path.exists():
        print(f"SKIP: {input_path} not found")
        return
    
    stats = {
        "total": 0,
        "zolai": 0,
        "burmese": 0,
        "english": 0,
        "mixed": 0,
        "cleaned_zolai": 0,
        "duplicates_removed": 0,
        "too_short": 0,
        "parse_errors": 0,
    }
    
    seen_hashes = set()
    
    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_zolai, "w", encoding="utf-8") as f_out:
        
        for line in f_in:
            line = line.strip()
            if not line:
                continue
            
            stats["total"] += 1
            
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                stats["parse_errors"] += 1
                continue
            
            # Classify language
            lang = classify_article(record)
            stats[lang] += 1
            
            # Keep only Zolai articles
            if lang != "zolai":
                continue
            
            # Clean content
            content = record.get("content", "")
            title = record.get("title", "")
            excerpt = record.get("excerpt", "")
            
            clean_content = clean_tongsan_text(content)
            clean_title = clean_tongsan_text(title)
            clean_excerpt = clean_tongsan_text(excerpt)
            
            if not clean_content or len(clean_content) < 50:
                stats["too_short"] += 1
                continue
            
            # Deduplicate by content hash
            content_hash = hashlib.md5(clean_content.encode("utf-8")).hexdigest()
            if content_hash in seen_hashes:
                stats["duplicates_removed"] += 1
                continue
            seen_hashes.add(content_hash)
            
            # Build clean record
            clean_record = {
                "id": record.get("id", ""),
                "date": record.get("date", ""),
                "slug": record.get("slug", ""),
                "title": clean_title,
                "excerpt": clean_excerpt,
                "content": clean_content,
                "language": "zolai",
                "source": "tongsan",
                "type": "article"
            }
            
            f_out.write(json.dumps(clean_record, ensure_ascii=False) + "\n")
            stats["cleaned_zolai"] += 1
    
    # Write stats
    with open(output_stats, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    
    print(f"\n=== Tongsan Cleaning Stats ===")
    print(f"Total articles: {stats['total']}")
    print(f"  Zolai: {stats['zolai']}")
    print(f"  Burmese: {stats['burmese']}")
    print(f"  English: {stats['english']}")
    print(f"  Mixed: {stats['mixed']}")
    print(f"\nCleaned Zolai articles: {stats['cleaned_zolai']}")
    print(f"Duplicates removed: {stats['duplicates_removed']}")
    print(f"Too short: {stats['too_short']}")
    print(f"Parse errors: {stats['parse_errors']}")

## 4. Run Cleaning Pipeline

In [ ]:
# Find all Tongsan article files
tongsan_files = list(INPUT_DIR.rglob("*tongsan*.jsonl"))

print(f"Found {len(tongsan_files)} Tongsan file(s)")

for tf in tongsan_files:
    print(f"\n{'='*60}")
    print(f"Processing: {tf.name}")
    print(f"{'='*60}")
    
    base_name = tf.stem.replace("tongsan_", "").replace("_articles", "")
    if not base_name:
        base_name = "all"
    
    out_zolai = OUTPUT_DIR / f"tongsan_zolai_clean_{base_name}.jsonl"
    out_stats = OUTPUT_DIR / f"tongsan_stats_{base_name}.json"
    
    process_tongsan_articles(tf, out_zolai, out_stats)

## 5. Merge All Cleaned Zolai Articles

In [ ]:
def merge_tongsan_zolai(output_dir: Path, merged_path: Path):
    """Merge all cleaned Zolai Tongsan articles into one file."""
    cleaned_files = list(output_dir.glob("tongsan_zolai_clean_*.jsonl"))
    
    total = 0
    seen = set()
    
    with open(merged_path, "w", encoding="utf-8") as f_out:
        for cf in sorted(cleaned_files):
            with open(cf, "r", encoding="utf-8") as f_in:
                for line in f_in:
                    line = line.strip()
                    if not line:
                        continue
                    
                    try:
                        record = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    
                    # Cross-file dedup by content hash
                    content = record.get("content", "")
                    h = hashlib.md5(content.encode("utf-8")).hexdigest()
                    if h in seen:
                        continue
                    seen.add(h)
                    
                    f_out.write(line + "\n")
                    total += 1
    
    print(f"Merged: {total} unique Zolai Tongsan articles")
    return total


merged_path = OUTPUT_DIR / "tongsan_zolai_clean_merged.jsonl"
merge_tongsan_zolai(OUTPUT_DIR, merged_path)

## 6. Validation & Samples

In [ ]:
# Validate merged output
if merged_path.exists():
    valid = 0
    invalid = 0
    has_burmese_count = 0
    has_html_count = 0
    
    with open(merged_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)
                valid += 1
                
                content = record.get("content", "")
                if has_burmese(content):
                    has_burmese_count += 1
                if re.search(r'<[^>]+>', content):
                    has_html_count += 1
            except json.JSONDecodeError:
                invalid += 1
    
    print(f"Validation Results:")
    print(f"  Valid JSONL lines: {valid}")
    print(f"  Invalid lines: {invalid}")
    print(f"  Still has Burmese: {has_burmese_count}")
    print(f"  Still has HTML: {has_html_count}")

In [ ]:
# Show sample cleaned articles
if merged_path.exists():
    print("=== Sample Cleaned Zolai Articles ===\n")
    with open(merged_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            record = json.loads(line)
            print(f"Article {i+1}: {record.get('title', 'N/A')[:80]}")
            print(f"Content preview: {record.get('content', '')[:300]}...")
            print("-" * 60)

In [ ]:
# Show length distribution
if merged_path.exists():
    lengths = []
    with open(merged_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            lengths.append(len(record.get("content", "")))
    
    if lengths:
        lengths.sort()
        print(f"Content length stats:")
        print(f"  Count: {len(lengths)}")
        print(f"  Min: {lengths[0]}")
        print(f"  Max: {lengths[-1]}")
        print(f"  Median: {lengths[len(lengths)//2]}")
        print(f"  Mean: {sum(lengths)//len(lengths)}")
        
        # Buckets
        buckets = Counter()
        for l in lengths:
            if l < 200:
                buckets["<200"] += 1
            elif l < 500:
                buckets["200-500"] += 1
            elif l < 1000:
                buckets["500-1000"] += 1
            elif l < 2000:
                buckets["1000-2000"] += 1
            else:
                buckets[">2000"] += 1
        
        print(f"\nLength buckets:")
        for k in ["<200", "200-500", "500-1000", "1000-2000", ">2000"]:
            print(f"  {k}: {buckets[k]}")